# 00 — Dataset Exploration

Quick look at the raw MELD CSVs: sizes, label distributions, text lengths, and missing values.
Nothing is modified here — this is purely observational.

> **Run this first** before any preprocessing notebook.

In [ ]:
import os
import sys

# Make sure the project root (parent of notebooks/) is on the path so 'src' is importable.
_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'src')) else os.path.abspath(os.path.join(_here, '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import pandas as pd
import matplotlib.pyplot as plt

from src.config import TRAIN_CSV, DEV_CSV, TEST_CSV, TEXT_COL, LABEL_COL, LABEL_REMAP

## Load raw splits

In [ ]:
train_raw = pd.read_csv(TRAIN_CSV)
dev_raw   = pd.read_csv(DEV_CSV)
test_raw  = pd.read_csv(TEST_CSV)

print('Columns:', train_raw.columns.tolist())
print(f'Train: {len(train_raw):,} rows | Dev: {len(dev_raw):,} rows | Test: {len(test_raw):,} rows')

## Missing values

In [ ]:
for name, df in [('train', train_raw), ('dev', dev_raw), ('test', test_raw)]:
    missing = df[[TEXT_COL, LABEL_COL]].isna().sum()
    print(f'[{name}] missing →', missing.to_dict())

## Label distribution

Note: MELD labels are lowercase; `joy` will be remapped to `happiness` later.

In [ ]:
def label_counts(df, split_name):
    counts = (
        df[LABEL_COL].astype(str).str.lower().str.strip()
        .replace(LABEL_REMAP)
        .value_counts()
        .sort_index()
    )
    print(f'\n--- {split_name} ---')
    print(counts.to_string())
    return counts

train_counts = label_counts(train_raw, 'train')
dev_counts   = label_counts(dev_raw,   'dev')
test_counts  = label_counts(test_raw,  'test')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, (counts, title) in zip(axes, [
    (train_counts, 'Train'), (dev_counts, 'Dev'), (test_counts, 'Test')
]):
    counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Emotion label distribution (MELD)', fontsize=13)
plt.tight_layout()
plt.show()

## Text length distribution

In [ ]:
train_raw['n_words'] = train_raw[TEXT_COL].astype(str).str.split().str.len()

print('Word count stats (train):')
print(train_raw['n_words'].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 3))
train_raw['n_words'].hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Word count')
ax.set_ylabel('Utterances')
ax.set_title('Utterance length distribution (train)')
plt.tight_layout()
plt.show()

## Sample utterances per emotion

In [ ]:
sample_df = (
    train_raw[[TEXT_COL, LABEL_COL]].copy()
    .assign(**{LABEL_COL: lambda d: d[LABEL_COL].str.lower().str.strip().replace(LABEL_REMAP)})
    .dropna()
    .groupby(LABEL_COL)
    .apply(lambda g: g.sample(min(3, len(g)), random_state=42))
    .reset_index(drop=True)
)
pd.set_option('display.max_colwidth', 120)
sample_df